# Code Review Assistant — LoRA Fine-Tuning on Mistral 7B

Fine-tune **Mistral-7B-Instruct** with **QLoRA** on real GitHub PR review data.

| Item | Value |
|------|-------|
| Base Model | `mistralai/Mistral-7B-Instruct-v0.3` |
| Dataset | [`ronantakizawa/github-codereview`](https://huggingface.co/datasets/ronantakizawa/github-codereview) |
| Method | 4-bit QLoRA + LoRA (PEFT + TRL SFTTrainer) |
| GPU | T4 (free Colab) or A100 (Kaggle) |
| Time | ~2-4 hours on T4 for 5K samples |

**Before running:** Runtime → Change runtime type → **T4 GPU**

In [1]:
# Install dependencies
!pip install -q datasets accelerate evaluate trl peft bitsandbytes transformers sacrebleu huggingface_hub

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — enable GPU runtime!'}")


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


GPU: CPU only — enable GPU runtime!


In [ ]:
from huggingface_hub import notebook_login

# Replace with your HuggingFace token (needs write access to push adapter)
HF_TOKEN = "hf_YOUR_TOKEN_HERE"  # @param {type:"string"}
HF_USERNAME = "your-username"  # @param {type:"string"}

notebook_login(token=HF_TOKEN)

BASE_MODEL = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = f"{HF_USERNAME}/code-review-mistral-lora"
MAX_SAMPLES = 5000  # @param {type:"integer"}
MAX_SEQ_LENGTH = 1024  # @param {type:"integer"}

In [ ]:
import json
from datasets import Dataset, load_dataset

SYSTEM = """You are a senior software engineer performing a code review.
Analyze the provided code and return a JSON array of review comments.
Each comment must have: type (bug|security|performance|style|suggestion|refactor|question), severity (high|medium|low), message, and optional line.
Return ONLY valid JSON."""

def format_example(row):
    comment_type = row.get("comment_type", "suggestion")
    severity = "high" if comment_type in ("bug", "security") else "medium" if comment_type == "performance" else "low"
    target = json.dumps([{
        "type": comment_type,
        "severity": severity,
        "message": row["reviewer_comment"],
        "line": row.get("comment_line") or None,
    }], ensure_ascii=False)
    user = f"""Language: {row.get('language', 'unknown')}
Code to review:
```
{row['before_code']}
```

Return a JSON array of review comments."""
    return {"text": f"<s>[INST] {SYSTEM}\n\n{user} [/INST] {target}</s>"}

# Option A: upload data/train.jsonl to Colab (Files panel) — faster startup
# Option B: stream from HuggingFace (default)
USE_PREPARED_DATA = False  # @param {type:"boolean"}

if USE_PREPARED_DATA:
    print("Loading prepared data/train.jsonl...")
    examples = []
    with open("train.jsonl", encoding="utf-8") as f:
        for line in f:
            examples.append(json.loads(line))
            if len(examples) >= MAX_SAMPLES:
                break
    dataset = Dataset.from_list(examples)
else:
    print("Loading github-codereview from HuggingFace...")
    raw = load_dataset("ronantakizawa/github-codereview", split="train", streaming=True)
    examples = []
    for row in raw:
        if row.get("is_negative") or not row.get("reviewer_comment"):
            continue
        examples.append(format_example(row))
        if len(examples) >= MAX_SAMPLES:
            break
    dataset = Dataset.from_list(examples)

split = dataset.train_test_split(test_size=0.05, seed=42)
train_dataset, eval_dataset = split["train"], split["test"]
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")
print("Sample:", train_dataset[0]["text"][:300], "...")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=HF_TOKEN)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
training_args = TrainingArguments(
    output_dir="./checkpoints",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_steps=200,
    save_total_limit=2,
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=0.3,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    peft_config=lora_config,
    processing_class=tokenizer,
    args=training_args,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Push LoRA adapter to HuggingFace Hub
trainer.model.push_to_hub(OUTPUT_DIR, token=HF_TOKEN)
tokenizer.push_to_hub(OUTPUT_DIR, token=HF_TOKEN)
print(f"Pushed adapter to: https://huggingface.co/{OUTPUT_DIR}")

## Before vs After Comparison

Run inference on the same code snippet with base model and fine-tuned adapter.

In [ ]:
from peft import PeftModel
from transformers import pipeline

TEST_CODE = '''def divide(a, b):
    return a / b'''

prompt = f"""<s>[INST] {SYSTEM}

Language: python
Code to review:
```
{TEST_CODE}
```

Return a JSON array of review comments. [/INST]"""

# Base model
base_pipe = pipeline("text-generation", model=BASE_MODEL, tokenizer=tokenizer,
                     max_new_tokens=256, do_sample=False, return_full_text=False)
base_out = base_pipe(prompt)[0]["generated_text"]

# Fine-tuned
ft_base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb_config, device_map="auto", token=HF_TOKEN)
ft_model = PeftModel.from_pretrained(ft_base, "./checkpoints")
ft_pipe = pipeline("text-generation", model=ft_model, tokenizer=tokenizer,
                   max_new_tokens=256, do_sample=False, return_full_text=False)
ft_out = ft_pipe(prompt)[0]["generated_text"]

print("=== BASE MISTRAL 7B ===")
print(base_out)
print("\n=== FINE-TUNED ===")
print(ft_out)

## Evaluation (BLEU-4)

After training, run `evaluation/evaluate.py` locally or copy the eval cell below.
Target: fine-tuned BLEU-4 > base BLEU-4 by 10+ points on held-out samples.